In [ ]:
!pip install -r requirements.txt

import os
from google.colab import userdata

os.environ["TABPFN_TOKEN"] = userdata.get("TABPFN_API")

import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from modelos import RandomForestWrapper, TabPFNWrapper, MitraWrapper

from sklearn.datasets import fetch_openml


def run_experiment(csv_path, target_col, model):
    print(f"\n Iniciando teste com modelo {model.__class__.__name__}")

    df = pd.read_csv(csv_path)
    train, test = train_test_split(df, test_size=0.25, random_state=42)

    start_fit = time.time()
    model.fit(train, target_col)
    fit_time = time.time() - start_fit

    start_predict = time.time()
    previsoes = model.predict(test, target_col)
    predict_time = time.time() - start_predict


    y_test = test[target_col]

    print("-" * 20)
    print(f"Tempo de treino: {fit_time:.2f} | Tempo de Predição: {predict_time:.2f}")
    print(f"MAE: {mean_absolute_error(y_test, previsoes):.4f}")
    print(f"RMSE: {mean_squared_error(y_test, previsoes)**0.5:.4f}")
    print(f"R2: {r2_score(y_test, previsoes):.4f}")

if __name__ == "__main__":


    dataset = fetch_openml(data_id=44015, as_frame=True, parser="auto")

    df = dataset.frame
    df.to_csv("credito.csv", index=False)



    target_col = "price" #mudar depois

    # Teste 1: Preço de casas usando Random Forest
    run_experiment(
        csv_path="credito.csv",
        target_col=target_col,
        model=RandomForestWrapper()
    )

    # Teste 2: Preço de casas usando TabPFN
    run_experiment(
        csv_path="credito.csv",
        target_col=target_col,
        model=TabPFNWrapper()
    )

    # Teste 3: Outro dataset totalmente diferente usando Mitra
    run_experiment(
        csv_path="credito.csv",
        target_col=target_col,
        model=MitraWrapper()
    )


 Iniciando teste com modelo RandomForestWrapper
--------------------
Tempo de treino: 44.92 | Tempo de Predição: 1.41
MAE: 0.1906
RMSE: 0.2440
R2: 0.9419

 Iniciando teste com modelo TabPFNWrapper


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.3
Python Version:     3.13.15
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          8
Pytorch Version:    2.11.0+cu130
CUDA Version:       13.0
GPU Memory:         GPU 0: 14.55/14.56 GB
Total GPU Memory:   Free: 14.55 GB, Allocated: 0.01 GB, Total: 14.56 GB
GPU Count:          1
Memory Avail:       47.51 GB / 50.99 GB (93.2%)
Disk Space Avail:   192.39 GB / 235.68 GB (81.6%)
No presets specified! To achieve strong results with AutoGluon, it is recommended to use the available presets. Defaulting to `'medium'`...
	Recommended Presets (For more details refer to https://auto.gluon.ai/stable/tutorials/tabular/tabular-essentials.html#presets):
	presets='extreme'  : Use this if you have a GPU. The go-to preset for best results, and the one to use for benchmark comparisons. New in v1.6: far better than 'best' on 

--------------------
Tempo de treino: 0.61 | Tempo de Predição: 171.09
MAE: 0.1782
RMSE: 0.2303
R2: 0.9483

 Iniciando teste com modelo MitraWrapper


Beginning AutoGluon training ...
AutoGluon will save models to "/content/modelo_mithra_temp"
Train Data Rows:    40455
Train Data Columns: 6
Label Column:       price
AutoGluon infers your prediction problem is: 'regression' (because dtype of label-column == float and many unique label-values observed).
	Label info (max, min, mean, stddev): (9.842622276764851, 5.7899601708972535, 7.78945, 1.01474)
	If 'regression' is not the correct problem_type, please manually specify the problem_type parameter during Predictor init (You may specify problem_type as one of: ['binary', 'multiclass', 'regression', 'quantile'])
Problem Type:       regression
Preprocessing data...
Using Feature Generators to preprocess the data ...
Fitting AutoMLPipelineFeatureGenerator...
	Available Memory:                    48646.89 MB
	Train Data (Original)  Memory Usage: 1.85 MB (0.0% of available memory)
	Inferring data type of each feature based on column values. Set feature_metadata_in to manually specify special 

RuntimeError: No models were trained successfully during fit(). Inspect the log output or increase verbosity to determine why no models were fit. Alternatively, set `raise_on_no_models_fitted` to False during the fit call.

In [ ]:
%%writefile modelos.py

import pandas as pd
import shutil
from sklearn.ensemble import RandomForestRegressor
from tabpfn import TabPFNRegressor
from autogluon.tabular import TabularPredictor

class RandomForestWrapper:
    def __init__(self):
        self.model = RandomForestRegressor(n_estimators=300, random_state=42)

    def fit(self, train_data, target_col):
        # Separa o x e o y
        X = train_data.drop(columns=[target_col])
        y = train_data[target_col]

        X = pd.get_dummies(X)
        self.feature_columns = X.columns

        self.model.fit(X,y)

    def predict(self, test_data, target_col):
        X = test_data.drop(columns=[target_col])

        X = pd.get_dummies(X)
        X = X.reindex(columns=self.feature_columns, fill_value=0)
        return self.model.predict(X)

class TabPFNWrapper:
    def __init__(self):
        self.model = TabPFNRegressor(ignore_pretraining_limits=True, random_state=42)

    def fit(self, train_data, target_col):
        X = train_data.drop(columns=[target_col])
        y = train_data[target_col]

        X = pd.get_dummies(X)
        self.feature_columns = X.columns

        self.model.fit(X, y)

    def predict(self, test_data, target_col):
        X = test_data.drop(columns=[target_col])
        X = pd.get_dummies(X)
        X = X.reindex(columns=self.feature_columns, fill_value=0)
        return self.model.predict(X, output_type="median")

class MitraWrapper:
    def __init__(self, path="modelo_mithra_temp", max_rows=50000):
        self.path = path
        self.max_rows = max_rows
        shutil.rmtree(self.path, ignore_errors=True)

    def fit(self, train_data, target_col):
        self.predictor = TabularPredictor(label=target_col, path=self.path)

        hyperparameters = {
            "MITRA": {
                "ag.max_rows": self.max_rows,  # Chave com prefixo 'ag.' que o AutoGluon valida
                "fine_tune": False,
            }
        }

        self.predictor.fit(
            train_data=train_data,
            hyperparameters=hyperparameters,
            # Permite utilizar mais margem de memória sem abortar preventivamente
            ag_args_fit={"ag.max_memory_usage_ratio": 2.0}
        )

    def predict(self, test_data, target_col):
        return self.predictor.predict(test_data).to_numpy()


Overwriting modelos.py
